# Lab 36 (solution): Training and hardening the router

Reference implementation. Replaces Lab 35's prompt-based classifier with a **trained** one (embeddings + logistic regression on a labeled query set), adds a **confidence gate**, and routes low-confidence queries to an **agentic fallback** that tries the top-2 candidate strategies and verifies before answering.

Builds directly on [Lab 35](../../35-adaptive-rag-router/). Same corpus (Lab 33), same eval set (Lab 34).

## Step 0: Setup

In [ ]:
import json
import os
import pathlib
import re
import time
import numpy as np
from dotenv import load_dotenv
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break
assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER = "openai"
MODEL = {"openai":"gpt-4o-mini","anthropic":"claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

In [ ]:
def chat(messages, temperature=0.0):
    if PROVIDER == "openai":
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return r.choices[0].message.content or ""
    from anthropic import Anthropic
    system = next((m["content"] for m in messages if m["role"]=="system"), "")
    ns = [m for m in messages if m["role"]!="system"]
    r = Anthropic().messages.create(model=MODEL, system=system, messages=ns, max_tokens=512, temperature=temperature)
    return "".join(b.text for b in r.content if hasattr(b,"text"))
def chat_token(messages, allowed):
    raw = chat(messages).strip().lower()
    for tok in allowed:
        if re.search(rf"\\b{re.escape(tok.lower())}\\b", raw):
            return tok
    return allowed[-1]

## Step 1: Labeled set + embeddings

81 queries labeled with one of five routes, embedded with the same model the retriever uses. Note the small-data leakage caveat in the cell.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

with open("./router_trainset.jsonl") as f:
    train = [json.loads(line) for line in f]
ROUTES = ["parametric","global","multihop","off_corpus_risk","specific"]
queries = [r["query"] for r in train]
labels  = [r["route"] for r in train]
X = embedder.encode(queries, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
y = np.array(labels)
from collections import Counter
print(f"{len(train)} labeled queries, {X.shape[1]}-dim embeddings")
for k,v in sorted(Counter(labels).items()):
    print(f"  {k}: {v}")
# NOTE on data leakage: several queries are near-paraphrases of each other. On a
# set this small a single random split is noisy and can leak a paraphrase across
# the train/test boundary, so we report stratified cross-validation, not one split.

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X, labels)


## Step 2: Train the classifier

Stratified cross-validation is the accuracy to trust on a set this small; the in-sample report is optimistic.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

clf = LogisticRegression(max_iter=1000, C=10.0, class_weight="balanced")
# Why logistic regression: small labeled set, want calibrated-ish predict_proba for
# the confidence gate, and an interpretable linear boundary. A linear SVM or small
# MLP are alternatives
# LR's probabilities are what the fallback gate needs.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(clf, X, y, cv=cv)
print(f"5-fold CV accuracy: {scores.mean():.2f} +/- {scores.std():.2f}")

clf.fit(X, y)  # fit on all data for downstream use
print("\nIn-sample report (optimistic - see CV above for the number to trust):")
print(classification_report(y, clf.predict(X), zero_division=0))

## Step 3: Confidence from `predict_proba`

The max class probability is the router's confidence and the input to the fallback gate.

In [ ]:
def route_with_confidence(query:
    str):
    """Trained router: embed -> LR -> (route, confidence, top2)."""
    v = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    proba = clf.predict_proba(v)[0]
    order = np.argsort(proba)[::-1]
    classes = clf.classes_
    return {"route": classes[order[0]], "confidence": float(proba[order[0]]),
            "top2": [(classes[i], float(proba[i])) for i in order[:2]]}

for q in ["Who leads the Helix Lab?",
          "Who is in charge of the lab focused on graph-based retrieval?",
          "What is cosine similarity?"]:
    r = route_with_confidence(q)
    print(f"{r['confidence']:.2f}  {r['route']:16} {q}")
    print(f"        top2: {r['top2']}")

## Step 4: Trained vs prompt router

Accuracy and cost against Lab 35's prompt classifier, on Lab 34's eval queries.

In [ ]:
# Compare the TRAINED classifier against the PROMPT-BASED router from Lab 35,
# on Lab 34's held-out eval queries. Two axes: accuracy and cost.

def classify_prompt(query):
    # the Lab 35 router, for comparison
    return chat_token([
        {"role":"system","content":"Classify for a RAG router. One label: parametric, "
         "global, multihop, specific, off_corpus_risk."},
        {"role":"user","content":query}], allowed=ROUTES)

CAT_TO_ROUTE = {"parametric":"parametric","global-theme":"global","multi-hop":"multihop",
                "off-corpus":"off_corpus_risk","specific-lookup":"specific","paraphrase":"specific"}
with open("../34-rag-pattern-head-to-head/eval_set.jsonl") as f:
    eval_set = [json.loads(line) for line in f]

t0=time.time()
trained_pred=[route_with_confidence(e["query"])["route"] for e in eval_set]
t_trained=time.time()-t0
t0=time.time()
prompt_pred =[classify_prompt(e["query"]) for e in eval_set]
t_prompt=time.time()-t0
truth=[CAT_TO_ROUTE[e["category"]] for e in eval_set]
def acc(predictions):
    return sum(a == b for a, b in zip(predictions, truth, strict=False)) / len(truth)
print(f"trained router:  acc={acc(trained_pred):.2f}   total_time={t_trained:.2f}s   (1 local embed + LR, no API cost)")
print(f"prompt router:   acc={acc(prompt_pred):.2f}   total_time={t_prompt:.2f}s   (1 LLM call per query, billed)")
print("\nThe trained router trades a one-time labeling+training cost for near-zero")
print("per-query cost and latency. The prompt router needs no labels but pays an LLM")
print("call every query and is harder to monitor for drift.")

## Step 5: Dispatch targets

The same compact strategies the router selects among (full versions in Labs 06/31/32/33).

In [ ]:
# Compact dispatch targets (full versions: Labs 06/31/32/33). Same as Lab 35.
CORPUS_DIR = pathlib.Path("../33-graph-rag-from-scratch/corpus")
chunks=[]
for p in sorted(CORPUS_DIR.glob("*.md")):
    if p.name == "README.md":
        continue
    body=p.read_text()
    for _i, para in enumerate(re.split(r"\n\s*\n", body)):
        if para.strip():
            chunks.append({"doc_id":p.stem,"text":para.strip()})
CE = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
def search(query,k=5):
    q=embedder.encode([query],normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)[0]
    s=CE@q
    return [chunks[i] for i in np.argsort(s)[::-1][:k]]
def _ground(query,k=5,allow_abstain=True):
    ctx="\n".join(f"[{c['doc_id']}] {c['text']}" for c in search(query,k))
    sys=("Answer using only the evidence; cite [doc ids]. "
         +("If the evidence lacks the answer, reply exactly 'INSUFFICIENT EVIDENCE'." if allow_abstain else ""))
    return chat([{"role":"system","content":sys},{"role":"user","content":f"Evidence:\n{ctx}\n\nQ: {query}"}])
def strat_parametric(q):
    return chat([{"role":"system","content":"Answer briefly from general knowledge."},{"role":"user","content":q}])
def strat_specific(q):
    return _ground(q,5)
def strat_graph(q):
    return _ground(q,8,allow_abstain=False)
def strat_corrective(q):
    v=chat_token([{"role":"system","content":"Do retrieved passages answer the query? one word: correct/incorrect."},
                  {"role":"user","content":q}],allowed=["correct","incorrect"])
    return "INSUFFICIENT EVIDENCE" if v=="incorrect" else _ground(q,5)
DISPATCH={"parametric":strat_parametric,"global":strat_graph,"multihop":strat_graph,
          "off_corpus_risk":strat_corrective,"specific":strat_specific}
print(f"{len(chunks)} chunks; dispatch targets ready")

## Step 6: Confidence gate + agentic fallback

High confidence dispatches directly; low confidence triggers a verified multi-strategy fallback.

In [ ]:
ABSTAIN=["insufficient evidence","does not contain","cannot answer","no information","don't have","do not have"]
def abstained(a):
    a = a.lower()
    return any(m in a for m in ABSTAIN)

def self_check(query, answer):
    """Cheap verifier: is the answer grounded/appropriate? Returns True/False."""
    if abstained(answer):
        return True   # abstaining is a valid, checkable outcome
    v=chat_token([{"role":"system","content":"Does the answer directly and specifically address "
                   "the question without hedging or contradiction? one word: yes or no."},
                  {"role":"user","content":f"Q: {query}\nA: {answer}"}],allowed=["yes","no"])
    return v=="yes"

def agentic_fallback(query, top2):
    """When the router is unsure, try the top-2 candidate strategies and keep the
    answer that passes the self-check
    if neither does, abstain. This spends extra
    calls ONLY on the uncertain tail."""
    tried=[]
    for route,_ in top2:
        ans=DISPATCH[route](query)
        ok=self_check(query,ans)
        tried.append((route,ans,ok))
        if ok and not abstained(ans):
            return {"answer":ans,"strategy":f"fallback->{route}","checked":True}
    # neither passed cleanly: prefer a clean abstention over a failed guess
    for route, ans, _ok in tried:
        if abstained(ans):
            return {"answer":ans,"strategy":f"fallback->{route} (abstained)","checked":True}
    return {"answer":tried[0][1],"strategy":f"fallback->{tried[0][0]} (low confidence)","checked":False}

def adaptive_rag_v2(query, threshold=0.50, verbose=True):
    r=route_with_confidence(query)
    if r["confidence"]>=threshold:
        ans=DISPATCH[r["route"]](query)
        out={"answer":ans,"strategy":r["route"],"confidence":r["confidence"],"fallback":False}
    else:
        fb=agentic_fallback(query, r["top2"])
        out={"answer":fb["answer"],"strategy":fb["strategy"],"confidence":r["confidence"],"fallback":True}
    if verbose:
        print(f"  conf={out['confidence']:.2f} {'FALLBACK ' if out['fallback'] else ''}-> {out['strategy']}")
    return out

## Step 7: When does the fallback help?

It fires only on the low-confidence tail, keeping average cost bounded.

In [ ]:
# How often does the fallback fire, and does it help on the uncertain tail?
THRESHOLD=0.50
fired=0
helped=[]
def correct(ans,item):
    if item["expected_behavior"]=="abstain":
        return abstained(ans)
    if abstained(ans):
        return False
    a=ans.lower()
    return all(t.lower() in a for t in item["expected_contains"])
for item in eval_set:
    r=route_with_confidence(item["query"])
    out=adaptive_rag_v2(item["query"], THRESHOLD, verbose=False)
    if out["fallback"]:
        fired+=1
        helped.append((item["query"][:44], correct(out["answer"],item), r["confidence"]))
print(f"Fallback fired on {fired}/{len(eval_set)} queries (confidence < {THRESHOLD}).")
for q,ok,c in helped:
    print(f"  conf={c:.2f} {'OK' if ok else 'XX'}  {q}")
print("\nThe gate is the point: the expensive multi-strategy fallback runs only on the")
print("low-confidence tail, so average cost stays close to the single-dispatch router")
print("while the hard queries get a careful, verified second look.")

## Step 8: Read the result

In [ ]:
# Tradeoffs to weigh:
#  - Trained vs prompt classifier: trained is ~free and fast per query and monitorable
#    for drift, but needs a labeled set and re-training as query distribution shifts.
#  - The classifier is still a single point of failure
#    the confidence gate + fallback
#    soften that by catching the queries the model is unsure about, not the ones it is
#    confidently wrong about (a calibration problem - watch for over-confident misroutes).
#  - The fallback costs multiple calls
#    the gate keeps that bounded to the tail.
print("Trained router + confidence gate + agentic fallback: cheaper steady state,")
print("a safety net for uncertainty, and a new thing to monitor - calibration.")

## What you built

A trained router (embeddings + logistic regression) with a confidence gate and an agentic fallback for uncertain queries. Versus Lab 35's prompt classifier: near-zero per-query cost and latency, monitorable for drift, at the price of a labeled set and retraining.

**Where this simplifies:** 81 labeled queries is small — cross-validation, not a single split, and real deployments need far more, refreshed as the query distribution shifts; logistic regression is a deliberate baseline (interpretable, calibrated-ish probabilities) rather than the strongest possible model; the fallback verifier is a single cheap LLM check, not a full agentic loop. The open risk is **calibration**: a confidently wrong route skips the gate, so monitor confidence against correctness, not just accuracy.

Next: [Lab 37](../../37-rag-eval-gates/) wraps this router in a CI eval gate and swaps token-presence scoring for an LLM judge.